In [ ]:
import re
import subprocess
import sys
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path(
    "/content/drive/MyDrive/data/jlens-reasoning/wheels"
)
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, "
        f"found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--requirement",
        str(REQUIREMENTS),
    ],
    check=True,
)
print(f"Installing project wheel {wheel.name}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--force-reinstall",
        "--no-deps",
        str(wheel),
    ],
    check=True,
)
print("Colab project installation complete")

del COMMIT_FILE, REQUIREMENTS, WHEEL_DIRECTORY, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(require_cuda=True)
context

In [ ]:
probe = context.runs_dir / "environment-check.txt"
probe.write_text("colab environment check\n", encoding="utf-8")

print(f"Commit: {PROJECT_COMMIT}")
print(f"Device: {context.device}")
print(f"Artifact root: {context.artifact_root}")
print(f"Drive write probe: {probe}")
print(f"W&B enabled: {context.wandb_enabled}")